# State-Space Time2Success Dataset Collection

Collects **successful-only** trajectories from the **final trained policy**,
capturing the observation vector (state space) + timestamp-derived
temporal-distance labels per frame. No vision, no video, no checkpoint
diversity — deliberately narrow scope matching what was asked. See the
reminders section at the end for what this scope leaves out.

## 1. Setup and sanity checks

In [ ]:
import gymnasium as gym
import metaworld
import numpy as np
import os, json
from stable_baselines3 import SAC

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
OBS_DIM = _probe.observation_space.shape[0]
_probe.close()

print("DT (seconds per step):", DT)
print("Observation dim:", OBS_DIM)  # confirms MLP input size before building anything

## 2. Load the final trained policy

In [ ]:
FINAL_CKPT = "./checkpoints/peg_insert_side/sac_peg_insert_final"
model = SAC.load(FINAL_CKPT)
print("Loaded:", FINAL_CKPT)

## 3. Collection function

Rolls out one episode deterministically, capturing `obs` at every step.
Returns `None` for the whole episode if it didn't succeed — filtering happens
at the call site, not here, so this function stays reusable if you ever want
failed episodes too.

In [ ]:
def collect_episode_state(model, env, deterministic=True, max_steps=500):
    obs, _ = env.reset()
    obs_list = []
    success_step = None
    for t in range(max_steps):
        obs_list.append(obs.copy())
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if terminated or truncated:
            break
    return obs_list, success_step

## 4. Smoke test — run before the full collection
Confirms the loop and labeling logic work on a handful of episodes first.

In [ ]:
N_SMOKE = 10
smoke_successes = 0
for i in range(N_SMOKE):
    env = make_env(seed=i)
    obs_list, success_step = collect_episode_state(model, env)
    env.close()
    if success_step is not None:
        smoke_successes += 1
        print(f"seed {i}: SUCCESS at step {success_step} ({success_step*DT:.2f}s), "
              f"obs shape check: {obs_list[0].shape}")
    else:
        print(f"seed {i}: failed (excluded)")

print(f"\nSmoke test success rate: {smoke_successes}/{N_SMOKE}")
print("If this is far off from the eval_history.json final entry, investigate before scaling up.")

## 5. Full collection — successful trajectories only

Since only successes are kept, and this uses a single (final, competent)
policy, most seeds should succeed — but not all. The loop below keeps
trying new seeds until it collects `TARGET_SUCCESSFUL_EPISODES`, rather than
running a fixed seed count and hoping enough succeed.

In [ ]:
# Number of successful episodes to collect
TARGET_SUCCESSFUL_EPISODES = 100  # start here; raise once the pipeline is confirmed working
MAX_SEED_ATTEMPTS = TARGET_SUCCESSFUL_EPISODES * 3  # safety cap in case success rate is low

episodes = []  # list of dicts: {obs_list, success_step, seed}
seed = 0
attempts = 0

while len(episodes) < TARGET_SUCCESSFUL_EPISODES and attempts < MAX_SEED_ATTEMPTS:
    env = make_env(seed=seed)
    obs_list, success_step = collect_episode_state(model, env)
    env.close()
    attempts += 1

    if success_step is not None:
        episodes.append(dict(obs_list=obs_list, success_step=success_step, seed=seed))
        if len(episodes) % 20 == 0:
            print(f"[{len(episodes)}/{TARGET_SUCCESSFUL_EPISODES}] collected "
                  f"(seed={seed}, attempts so far={attempts})")

    seed += 1

print(f"\nDone: {len(episodes)} successful episodes from {attempts} attempts "
      f"(observed success rate: {len(episodes)/attempts:.2%})")
if len(episodes) < TARGET_SUCCESSFUL_EPISODES:
    print("WARNING: hit MAX_SEED_ATTEMPTS before reaching target — "
          "either raise the cap or lower the target.")

## 6. Build (state, label) rows with temporal-distance labels

Two label units kept side by side (steps and seconds) — use whichever
the training script expects; steps is the more numerically stable target,
seconds is more interpretable.

In [ ]:
records = []

for ep_idx, ep in enumerate(episodes):
    success_step = ep["success_step"]
    for t, obs in enumerate(ep["obs_list"][:success_step + 1]):
        records.append(dict(
            episode_idx=ep_idx,
            seed=ep["seed"],
            frame_idx=t,
            timestamp_sec=round(t * DT, 4),
            steps_remaining=success_step - t,
            seconds_remaining=round((success_step - t) * DT, 4),
            obs=obs.tolist(),  # keep as list for JSON-safety; convert back to array at load time
        ))

print(f"Total (state, label) rows: {len(records)}")
print(f"From {len(episodes)} successful episodes")

## 7. Save the dataset

In [ ]:
import numpy as np

OUTPUT_DIR = "./data/state_space_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save as numpy arrays for fast loading in the training notebook
X = np.array([r["obs"] for r in records], dtype=np.float32)
y_steps = np.array([r["steps_remaining"] for r in records], dtype=np.float32)
y_seconds = np.array([r["seconds_remaining"] for r in records], dtype=np.float32)
episode_ids = np.array([r["episode_idx"] for r in records], dtype=np.int32)  # for group-aware splitting

np.savez(
    os.path.join(OUTPUT_DIR, "dataset.npz"),
    X=X, y_steps=y_steps, y_seconds=y_seconds, episode_ids=episode_ids,
)

# Also save a human-readable index/summary
summary = dict(
    total_rows=len(records),
    total_episodes=len(episodes),
    obs_dim=int(OBS_DIM),
    dt_sec=DT,
    source_policy=FINAL_CKPT,
    successful_only=True,
    checkpoint_diversity=False,
)
with open(os.path.join(OUTPUT_DIR, "dataset_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved to:", os.path.abspath(OUTPUT_DIR))
print(summary)

## 8. Reminders — what this scope leaves out

Read before treating this dataset as final:

- **Successful-only, single-policy source.** No failure/regression coverage
  (unlike the earlier checkpoint-diversity discussion) — this dataset
  will only ever describe states along near-optimal trajectories. Fine for
  a first pass, but revisit if the trained regressor needs to handle
  off-distribution states later (e.g., for reward shaping a *different*,
  less-mature policy).
- **`obs` composition not yet verified.** Step 1 prints `OBS_DIM` but doesn't
  confirm whether velocity is included, only position — worth checking
  MetaWorld's docs/source for this task if the MLP-vs-RNN question ever
  comes back up.
- **`TARGET_SUCCESSFUL_EPISODES=100` is a starting guess**, not a validated
  sufficient size — same caution as the "small amount of data" discussion
  earlier: treat a first training run on this as a pipeline smoke test, not
  a final result.
- **Group-aware splitting still needed at training time** —
  `episode_ids` is saved specifically so the training script can split by
  episode, not by row, avoiding the leakage issue flagged in stage 2.
- **The stage-2 training script assumed the video/embedding dataset format**
  — it needs updating to load this `.npz` format and use a plain MLP on
  `X` directly instead of the frozen-encoder embedding step. Worth doing as
  the next piece.